# 18 · UDFs: when (not) to use them (Spark)

A **UDF** (user-defined function) runs your own Python code on each value.

- A plain **Python UDF** sends every row from the JVM (where Spark runs) to a Python process and back:
  slow, and Spark cannot optimize inside it.
- A **pandas UDF** (vectorized) sends whole batches with Apache Arrow: much faster.
- **Built-in functions** (`F.*`) are fastest: always try them first.

Interview answer: "I avoid Python UDFs; I use built-ins, then pandas UDFs if I really need Python."

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Python UDF vs built-in  ·  🟢 easy

Make a column `name_upper` two ways: a Python UDF and `F.upper`. Show both on 3 rows and print both plans. Find `BatchEvalPython` in the UDF plan.

In [ ]:
# your answer here (DataFrame API)

## Q2. Handle NULL inside a UDF  ·  🟢 easy

Write a UDF `city_code` that returns the first 3 letters of a city in capitals. Run it on `customers` **without** a NULL check first (it fails on the NULL city), then fix it.

*Expected result: 2 rows · columns: `customer_id`, `city`, `code`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q3. Wrong return type gives silent NULLs  ·  🟡 medium

Declare a UDF that returns `int` but really returns a float (salary / 1000). What do you get? Fix the declared type.

*Expected result: 2 rows · columns: `salary`, `k`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q4. Pandas UDF (vectorized)  ·  🟡 medium

Write a pandas UDF `with_tax` that multiplies a price Series by 1.18 and rounds to 2 decimals. Apply it to `products`.

*Expected result: 5 rows · columns: `product_name`, `price`, `price_with_tax`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q5. Use a UDF in Spark SQL  ·  🟢 easy

Register a Python function as `initials` for SQL with `spark.udf.register`, then call it in `%%sql`.

*Expected result: 3 rows · columns: `name`, `ini`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

In [ ]:
%%sql
-- your answer here

## Q6. Replace a UDF with built-ins  ·  🟡 medium

This UDF labels orders: amount NULL → `missing`, > 200 → `big`, else `normal`. Write the same with `F.when` and check that both give the same counts per label.

*Expected result: 3 rows · columns: `label`, `count`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q7. Per-group pandas logic: applyInPandas  ·  🔴 hard

For each department, use pandas to compute each employee's salary **z-score** ((salary − mean) / std) with `groupBy(...).applyInPandas`. Skip NULL dept_id.

*Expected result: 29 rows · columns: `dept_id`, `name`, `z`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()